<a href="https://colab.research.google.com/github/titan-spyer/Bold-fitness/blob/main/CloudNotebooks/ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)

In [5]:
df = pd.read_csv('fmnist_small.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'fmnist_small.csv'

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle("First 16 Images", fontsize=16)

for i, ax in enumerate(axes.flat):
  img = df.iloc[1, 1:].values.reshape(28, 28)
  ax.imshow(img)
  ax.axis('off')
  ax.set_title(f"Lable: {df.iloc[i, 0]}")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
x = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
x_train = x_train/255.0
x_test = x_test/255.0  # Commented out as x_test is an empty list

In [ ]:
x_train

In [ ]:
# Create Custom dataset class
class CustomDataset(Dataset):
  def __init__(self, features, lables):
    self.features = torch.tensor(features, dtype=torch.float32)
    self.lables = torch.tensor(lables, dtype=torch.long)

  def __len__(self):
    return len(self.features)


  def __getitem__(self, index):
    return self.features[index], self.lables[index]

In [ ]:
train_dataset = CustomDataset(x_train, y_train)

In [ ]:
train_dataset[0]

In [ ]:
# Create Test Dataset Object
test_datset = CustomDataset(x_test, y_test)

In [ ]:
# Create train and test loader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_datset, batch_size=32, shuffle=False)

In [6]:
# Define NN class
class MyNeuralNetwork(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        # nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(128, 64),
        # nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )

  def forward(self, x):
    return self.model(x)

In [ ]:
# Set Learning rate and epochs
epochs = 100
learning_rate = 0.1

In [ ]:
# Instatiate the model
model = MyNeuralNetwork(x_train.shape[1])
model = model.to(device)
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [ ]:
# Training Loop

for epoch in range(epochs):
  for batch_features, batch_lables in train_loader:
    batch_features = batch_features.to(device)
    batch_lables = batch_lables.to(device)
    # forward pass
    outputs = model(batch_features)
    # Caclculate loss
    loss = criterion(outputs, batch_lables)

    # clear the gradients
    optimizer.zero_grad()

    # Backward pass
    loss.backward()

    # Update grads
    optimizer.step()

In [ ]:
# Set Model to eval mode
model.eval()

In [ ]:
# Evaluation code
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_lables in test_loader:
    batch_features = batch_features.to(device)
    batch_lables = batch_lables.to(device)
    outputs = model(batch_features)
    _, predicted = torch.max(outputs, 1)
    total += batch_lables.shape[0]
    correct += (predicted == batch_lables).sum().item()

print(correct/total)